In [0]:
import time
import requests

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    BooleanType,
)

from pyspark.sql.functions import current_timestamp
APP_ID = dbutils.secrets.get(
    scope="adzuna",
    key="app-id",
)

APP_KEY = dbutils.secrets.get(
    scope="adzuna",
    key="app-key",
)

BASE_URL = "https://api.adzuna.com/v1/api/jobs/us/search"

SEARCH_ROLES = [
    "data engineer",
    "data scientist",
    "data analyst",
    "business analyst",
    "financial analyst",
    "analytics engineer",
    "machine learning engineer",
    "machine learning scientist",
    "software engineer",
    "ai engineer",
    "forward deployment engineer",
    "computer vision engineer",
    "business intelligence engineer",
    "bi analyst",
    "business analyst",
    "financial analyst",
    "data visualization engineer",
    "data architect",
]

SEARCH_LOCATIONS = [
    "Dallas, TX",
    "Austin, TX",
    "Houston, TX",
    "Chicago, IL",
    "New York, NY",
    "Seattle, WA",
    "Boston, MA",
    "Atlanta, GA",
    "Denver, CO",
    "Portland, OR",
    "San Francisco, CA",
    "Los Angeles, CA",
    "Philadelphia, PA",
    "Miami, FL",
    "Phoenix, AZ",
    "San Diego, CA",
    "San Jose, CA",
    "Detroit, MI",
    "Milwaukee, WI",
    "Columbus, OH",
    "Minneapolis, MN",    
]


PAGES_PER_SEARCH = 1
RESULTS_PER_PAGE = 50

MAX_RETRIES = 4
REQUEST_DELAY_SECONDS = 1.5


all_jobs = []
request_count = 0


for role in SEARCH_ROLES:
    for search_location in SEARCH_LOCATIONS:
        print("=" * 70)
        print(f"Role: {role}")
        print(f"Location: {search_location}")
        print("=" * 70)
        for page in range(1, PAGES_PER_SEARCH + 1):
            url = f"{BASE_URL}/{page}"
            params = {
                "app_id": APP_ID,
                "app_key": APP_KEY,
                "results_per_page": RESULTS_PER_PAGE,
                "what": role,
                "where": search_location,
            }
            page_success = False


            for attempt in range(MAX_RETRIES):
                try:
                    response = requests.get(url, params=params,timeout=30,)
                    request_count += 1
                    print(f"Page {page} status: " f"{response.status_code}")

                    if response.status_code in (429,500,502,503,504,):
                        wait_seconds = (2 ** attempt)
                        print("Temporary API error. "f"Retrying in "f"{wait_seconds} seconds...")
                        time.sleep(wait_seconds)
                        continue
                    response.raise_for_status()

                    data = response.json()
                    page_jobs = data.get("results",[],)
                    print("Jobs retrieved: " f"{len(page_jobs)}")
                    all_jobs.extend(page_jobs)
                    page_success = True

                    if (len(page_jobs) < RESULTS_PER_PAGE):
                        print("Last available page " "reached for this search.")
                    break
                except (requests.exceptions.RequestException) as e:
                    if (attempt == MAX_RETRIES - 1):
                        print("Request failed permanently for " f"{role} / " f"{search_location} / " f"page {page}: " f"{e}")

                    else:
                        wait_seconds = (2 ** attempt)
                        print("Request error. " f"Retrying in "f"{wait_seconds} seconds...")
                        time.sleep(wait_seconds)

            if not page_success:
                print("Skipping remaining pages " "for this role/location.")
                break

            if (len(page_jobs) < RESULTS_PER_PAGE):
                break

            time.sleep(REQUEST_DELAY_SECONDS)

print("=" * 70)

print("API requests made:", request_count,)
print("Total jobs retrieved " "before deduplication:", len(all_jobs),)

unique_jobs = {}
for job in all_jobs:
    job_id = job.get("id")
    if job_id is None:
        continue
    unique_jobs[str(job_id)] = job
jobs = list(unique_jobs.values())
print("Unique jobs after " "deduplication:", len(jobs),)

if not jobs:
    raise ValueError("No jobs were retrieved from Adzuna. " "Check API credentials, API availability, " "and search settings.")


schema = StructType([
    StructField("job_id",StringType(),True,),
    StructField("title",StringType(),True,),
    StructField("description",StringType(),True,),
    StructField("company",StringType(),True,),
    StructField("location",StringType(),True,),
    StructField("category",StringType(),True,),
    StructField("salary_min",DoubleType(),True,),
    StructField("salary_max",DoubleType(),True,),
    StructField("salary_is_predicted",BooleanType(),True,),
    StructField("latitude",DoubleType(),True,),
    StructField("longitude",DoubleType(),True,),
    StructField("job_url",StringType(),True,),
    StructField("created",StringType(),True,),
])

rows = []
for job in jobs:
    company = (job.get("company") or {})
    location = (job.get("location") or {})
    category = (job.get("category") or {})
    salary_min = (float(job["salary_min"])
        if job.get("salary_min") is not None
        else None
    )
    salary_max = (float(job["salary_max"])
        if job.get("salary_max") is not None
        else None
    )
    salary_is_predicted = (bool(job["salary_is_predicted"])
        if job.get("salary_is_predicted") is not None
        else None
    )
    latitude = (float(job["latitude"])
        if job.get("latitude") is not None
        else None
    )
    longitude = (float(job["longitude"])
        if job.get("longitude") is not None
        else None
    )
    rows.append(
        (
            str(job.get("id"))
            if job.get("id")
            else None,
            
            job.get("title"),
            job.get("description"),
            company.get("display_name"),
            location.get("display_name"),
            category.get("display_name"),
            salary_min,
            salary_max,
            salary_is_predicted,
            latitude,
            longitude,
            job.get("redirect_url"),
            job.get("created"),
        )
    )

jobs_df = spark.createDataFrame(rows, schema=schema,)

jobs_df = jobs_df.withColumn("pipeline_run_time", current_timestamp(),)

(
    jobs_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("career_os.bronze.jobs_raw")
)

print("Bronze ingestion complete.")
print(f"Rows written: " f"{jobs_df.count()}")
print("Table: " "career_os.bronze.jobs_raw")